# Lo que los mapas de inundación no vieron en Valencia

## Objetivo

Quiero comprobar si la cartografía oficial anticipó por igual la huella observada de la DANA del 29 de octubre de 2024 en todos los municipios valencianos. Este trabajo es un **EDA**: describo y comparo datos ya existentes; no predigo futuras inundaciones.

Parto de los agregados municipales publicados por EasyxLab a partir de Copernicus EMS, Catastro, SNCZI y PATRICOVA. No presento sus cálculos como propios. Mi aportación consiste en estudiar cómo se distribuye territorialmente el desfase y cuánto se concentra en unos pocos municipios.


## 1. Preparación de los datos

Separo esta fase del análisis. Conservo los textos como `<5` cuando la fuente ha ocultado celdas pequeñas para proteger datos. Solo uso municipios con cifras publicadas en las comparaciones municipales.


In [1]:
from pathlib import Path
import csv
import json

BASE = Path('datos')

with (BASE / 'municipios.csv').open(encoding='utf-8-sig', newline='') as f:
    municipios = list(csv.DictReader(f))

with (BASE / 'resumen_fuente.json').open(encoding='utf-8') as f:
    resumen = json.load(f)

def entero(valor):
    return int(valor) if valor and valor.isdigit() else None

def decimal(valor):
    try:
        return float(valor)
    except (TypeError, ValueError):
        return None

filas = []
for fila in municipios:
    viviendas = entero(fila['dwellings_in_extent'])
    fuera = entero(fila['dwellings_outside_all_zones'])
    proporcion = decimal(fila['share_outside_all_zones'])
    if viviendas and fuera is not None and proporcion is not None:
        filas.append({
            'municipio': fila['municipality'],
            'viviendas_huella': viviendas,
            'viviendas_fuera': fuera,
            'proporcion_fuera': proporcion,
        })

print(f'Municipios del archivo: {len(municipios)}')
print(f'Municipios comparables sin celdas ocultas: {len(filas)}')


Municipios del archivo: 121
Municipios comparables sin celdas ocultas: 47


## 2. Resultado provincial

Uso como escenario central la regla de referencia de la fuente: la huella de Copernicus intersecta el edificio y el edificio toca la envolvente de la zona oficial. La envolvente cierra huecos secos de hasta dos hectáreas que dejan las capas de profundidad alrededor de edificios.


In [2]:
principal = resumen['primary']
total = principal['dwellings_in_extent']
fuera = principal['dwellings_by_status']['outside every official zone']
porcentaje = 100 * principal['share_outside_all_zones']

print(f'Viviendas dentro de la huella: {total:,}'.replace(',', '.'))
print(f'Viviendas fuera de todas las zonas oficiales: {fuera:,}'.replace(',', '.'))
print(f'Proporción: {porcentaje:.1f} %'.replace('.', ','))


Viviendas dentro de la huella: 84.789
Viviendas fuera de todas las zonas oficiales: 43.359
Proporción: 51,1 %


La cifra provincial es útil, pero oculta dos decisiones importantes. La huella de Copernicus y la elaborada por la Generalitat no delimitan exactamente la misma inundación. Además, el resultado cambia si exijo que el centro del edificio, y no cualquier parte de su planta, quede dentro de una zona. Por eso trato el 51,1 % como escenario central y no como una medida exacta sin incertidumbre.


## 3. Las diferencias entre municipios

Ordeno por número de viviendas situadas fuera de todas las zonas oficiales. Esto evita que un municipio pequeño con un porcentaje extremo domine la historia.


In [3]:
ordenadas = sorted(filas, key=lambda x: x['viviendas_fuera'], reverse=True)

print('Municipio                    Viviendas fuera    Parte de su huella')
for fila in ordenadas[:12]:
    nombre = fila['municipio'][:27]
    print(f"{nombre:<28} {fila['viviendas_fuera']:>14,} {fila['proporcion_fuera']:>17.1%}".replace(',', '.'))


Municipio                    Viviendas fuera    Parte de su huella
Paiporta                              8.619             71.9%
València                              6.795             81.0%
Picanya                               4.674             95.8%
Alberic                               4.590             93.7%
Xirivella                             4.466             37.7%
Castelló                              2.419             73.3%
la Pobla Llarga                       1.936             82.2%
Sedaví                                1.878            100.0%
Alfafar                               1.414             49.8%
Torrent                               1.413             92.2%
Aldaia                                  841             24.3%
Rafelguaraf                             669             64.1%


In [4]:
top3 = sum(f['viviendas_fuera'] for f in ordenadas[:3])
top5 = sum(f['viviendas_fuera'] for f in ordenadas[:5])

print(f'Las tres primeras reúnen {top3:,} viviendas: {top3/fuera:.1%} del total provincial fuera de zona.'.replace(',', '.'))
print(f'Las cinco primeras reúnen {top5:,} viviendas: {top5/fuera:.1%} del total provincial fuera de zona.'.replace(',', '.'))


Las tres primeras reúnen 20.088 viviendas: 46.3% del total provincial fuera de zona.
Las cinco primeras reúnen 29.144 viviendas: 67.2% del total provincial fuera de zona.


La mitad provincial no describe una experiencia común. En Paiporta, València y Picanya se concentra casi la mitad de las viviendas del análisis situadas fuera de todas las zonas oficiales. Al mismo tiempo, otros municipios presentan una cobertura muy alta. La cartografía no falló de una sola manera ni con la misma intensidad.


## 4. Dos extremos que cuentan historias opuestas

Calculo cuántos municipios comparables quedan en los extremos: tres cuartas partes o más de sus viviendas fuera de zona, frente a una cuarta parte o menos.


In [5]:
muy_fuera = [f for f in filas if f['proporcion_fuera'] >= 0.75]
muy_cubiertos = [f for f in filas if f['proporcion_fuera'] <= 0.25]

print(f'Municipios con al menos el 75 % fuera: {len(muy_fuera)}')
print(f'Municipios con como máximo el 25 % fuera: {len(muy_cubiertos)}')
print()
print('Al menos el 75 % fuera:')
for f in sorted(muy_fuera, key=lambda x: x['proporcion_fuera'], reverse=True):
    print(f"- {f['municipio']}: {f['proporcion_fuera']:.1%} ({f['viviendas_fuera']:,} viviendas)".replace(',', '.'))


Municipios con al menos el 75 % fuera: 12
Municipios con como máximo el 25 % fuera: 21

Al menos el 75 % fuera:
- Catadau: 100.0% (30 viviendas)
- Llocnou de la Corona: 100.0% (62 viviendas)
- Sant Joanet: 100.0% (253 viviendas)
- Sedaví: 100.0% (1.878 viviendas)
- Sinarcas: 100.0% (8 viviendas)
- Picanya: 95.8% (4.674 viviendas)
- Alberic: 93.7% (4.590 viviendas)
- Torrent: 92.2% (1.413 viviendas)
- Sueca: 91.8% (491 viviendas)
- la Pobla Llarga: 82.2% (1.936 viviendas)
- València: 81.0% (6.795 viviendas)
- Benicull de Xúquer: 80.8% (597 viviendas)


## 5. Sensibilidad y límites

La fuente sitúa la proporción fuera de toda zona entre el 38,9 % y el 61,9 % al combinar dos huellas de inundación y tres reglas razonables. Con variantes adicionales llega hasta el 66,0 %. Esta amplitud no es un error estadístico: nace de cómo se dibuja la inundación y de qué significa que un edificio esté dentro de una zona.

La limitación más delicada es temporal. La capa estatal consultada por EasyxLab figura actualizada el 20 de diciembre de 2024, después de la DANA. La versión vigente el 29 de octubre estaba fechada en 2022 y no pudo recuperarse. Por tanto, el análisis no demuestra exactamente qué sabía el mapa estatal el día de la riada; probablemente concede a la cartografía anterior una ventaja, pero no puedo cuantificarla.


In [6]:
rangos = resumen['ranges']
print('Escenarios centrales: {:.1%}–{:.1%}'.format(*rangos['headline_dwellings']))
print('Incluyendo variantes: {:.1%}–{:.1%}'.format(*rangos['with_variants_dwellings']))
print('Al retirar un municipio cada vez: {:.1%}–{:.1%}'.format(*rangos['leave_one_out']))


Escenarios centrales: 38.9%–61.9%
Incluyendo variantes: 38.9%–66.0%
Al retirar un municipio cada vez: 47.7%–54.9%


## 6. Conclusión

Los datos no sostienen una frase simple como «Valencia sabía dónde se inundaría». En el escenario central, algo más de la mitad de las viviendas situadas dentro de la huella de Copernicus quedaba fuera de todas las zonas oficiales consultadas. Pero el dato decisivo es territorial: algunos municipios estaban casi enteros dentro de las previsiones y otros apenas aparecían.

La revisión anunciada en 2026 amplía un 61,2 % la superficie peligrosa del área de la DANA, de 27.162 a 43.779 hectáreas. Esa cifra muestra que la administración ha redibujado el riesgo, pero la cartografía detallada revisada todavía no estaba disponible como capa descargable durante este análisis. No comparo polígonos que no he podido obtener.

## Fuentes

- EasyxLab, estudio S19 y agregados municipales, CC BY 4.0.
- Copernicus Emergency Management Service, activación EMSR773.
- Generalitat Valenciana, PATRICOVA y huella DANA 2024.
- MITECO, Sistema Nacional de Cartografía de Zonas Inundables.
- Dirección General del Catastro, INSPIRE Buildings.
- IGN, límites municipales.
